# 02 · Material classification (milestone M2)

**Interim**: four classes from Sentinel-2 (vegetation / dark / bright / bare soil), per the fallback in CLAUDE.md. Replaced by the EnMAP classifier when data access is granted. Validate by spot-checking 20 roofs against high-resolution imagery.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from coolcairo.config import load_config, geobox_for
cfg = load_config()

In [ ]:
from coolcairo.pipeline import load_sentinel2
from coolcairo.classify import classify_sentinel2, Material
s2 = load_sentinel2(cfg)
material = classify_sentinel2(cfg, s2)
shares = material.to_series().value_counts(normalize=True).rename(lambda c: Material(c).name)
shares

In [ ]:
from matplotlib.colors import ListedColormap
cmap = ListedColormap(['#999999', '#4f8a3c', '#38383d', '#dbd4c2', '#d6b37d'])
material.plot(cmap=cmap, vmin=-0.5, vmax=4.5, figsize=(9, 8))
plt.title('Interim material classes: nodata / vegetation / dark / bright / soil')

## Bare soil threshold (`bsi_soil_min`)
Brightness cannot separate pale roofs from desert soil. The Bare Soil Index threshold is chosen as the best single split between pixels inside OSM footprints (roofs) and open ground more than 300 m from any footprint. Re-running this cell reproduces the value in `config/aoi.yaml`.

In [ ]:
from scipy.ndimage import distance_transform_edt
from coolcairo.buildings import footprint_mask
from coolcairo.classify import soil_threshold
from coolcairo.pipeline import load_buildings, FINE_RESOLUTION
roof = footprint_mask(load_buildings(cfg), geobox_for(cfg, 'model_aoi', FINE_RESOLUTION)).values.astype(bool)
open_ground = distance_transform_edt(~roof) * FINE_RESOLUTION > 300
threshold, accuracy = soil_threshold(s2, roof, open_ground, cfg['s2_classification']['ndvi_vegetation'])
print(f'BSI threshold {threshold:.3f}, balanced accuracy {accuracy:.2f}; config uses {cfg["s2_classification"]["bsi_soil_min"]}')

## Spot-check sheet (M2)
20 points, 5 per class, inside the display district and only in uniform 30 m patches (so a small offset cannot land on another class). A person fills in `docs/m2_spot_check.xlsx` from Google Maps satellite view; the agreement figure goes in the report. Fixed seed, so re-running produces the same points.

In [ ]:
from coolcairo import validation
from coolcairo.config import REPO_ROOT
points = validation.sample_points(cfg, material)
sheet = REPO_ROOT / 'docs' / 'm2_spot_check.xlsx'
if not sheet.exists():  # never overwrite a sheet someone has started filling in
    validation.write_sheet(points, sheet)
points[['lat', 'lon', 'predicted_label']].round(5)

In [ ]:
# After the sheet is filled in: agreement per point and overall.
results = validation.read_results(sheet)
print(f"Agreement: {results.match.mean():.0%} of {int(results.match.notna().sum())} checked points")
results